## SECTION 1 — Install Required Libraries

In [2]:
# Install necessary libraries
!pip install supabase pandas numpy scikit-learn joblib fastapi uvicorn pyngrok nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 1.5 MB/s eta 0:00:00


## SECTION 2 — Connect to Supabase

In [3]:
from supabase import create_client, Client
from google.colab import userdata
import os

# Your Supabase project URL and service role key
# IMPORTANT: Use your `Service Role Key` for full access, not the `Anon Public Key`.
# You can find your Service Role Key in Supabase Project Settings -> API.
SUPABASE_URL = "https://xdcsgcmgwlyndxrbkcav.supabase.co"
# It is recommended to store your Supabase Key in Colab Secrets
SUPABASE_KEY = userdata.get('SUPABASE_SERVICE_ROLE_KEY') or os.environ.get('SUPABASE_SERVICE_ROLE_KEY')

if not SUPABASE_KEY:
    print("Warning: SUPABASE_SERVICE_ROLE_KEY not found in Colab Secrets or environment variables.\nPlease add it for full functionality.")

supab_client: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

print("Supabase client created successfully.")

Supabase client created successfully.


## SECTION 3 — Fetch Live Data From Supabase

We will define a function to fetch data from the required tables. Initially, these tables might be empty. For demonstration purposes in the FastAPI section, we will assume some structure for feature generation. You should populate these tables in your Supabase project for real-world usage.

In [4]:
import pandas as pd

def fetch_data_from_supabase(table_name):
    try:
        response = supab_client.from_(table_name).select("*", count='exact').execute()
        if response.data:
            print(f"Fetched {len(response.data)} records from {table_name}.")
            return pd.DataFrame(response.data)
        else:
            print(f"No data found in table: {table_name}")
            return pd.DataFrame()
    except Exception as e:
        print(f"Error fetching data from {table_name}: {e}")
        return pd.DataFrame()

# Fetch data from each table
incidents_df = fetch_data_from_supabase('incidents')
road_segments_df = fetch_data_from_supabase('road_segments')
vehicles_df = fetch_data_from_supabase('vehicles')
traffic_signals_df = fetch_data_from_supabase('traffic_signals')

print("\n--- Sample Data (if available) ---")
if not incidents_df.empty:
    print("Incidents:")
    display(incidents_df.head())
if not road_segments_df.empty:
    print("\nRoad Segments:")
    display(road_segments_df.head())
if not vehicles_df.empty:
    print("\nVehicles:")
    display(vehicles_df.head())
if not traffic_signals_df.empty:
    print("\nTraffic Signals:")
    display(traffic_signals_df.head())

No data found in table: incidents
No data found in table: road_segments
No data found in table: vehicles
No data found in table: traffic_signals

--- Sample Data (if available) ---


## SECTION 4 — Define Feature Structure

The Machine Learning model will use the following features for prediction:

-   **`incident_count`**: Number of incidents where `resolved_at` is `NULL`.
-   **`avg_speed`**: Average of `road_segments.current_speed`.
-   **`heavy_congestion`**: Number of road segments where `congestion == "heavy"`.
-   **`signals`**: Total number of traffic signals.
-   **`emergency_density`**: Number of vehicles where `type` is 'ambulance', 'fire', or 'police'.

## SECTION 5 — Generate Synthetic Training Data

Since the Supabase tables are likely empty or lack historical data, we will generate synthetic data for training the ML model. This ensures we have enough data (at least 2000 rows) with realistic feature distributions.

In [5]:
import numpy as np
import pandas as pd

np.random.seed(42) # for reproducibility

num_rows = 2500 # At least 2000 rows

# Generate synthetic data for features
synthetic_data = {
    'incident_count': np.random.randint(0, 6, num_rows), # 0-5
    'avg_speed': np.random.randint(10, 71, num_rows), # 10-70 km/h
    'heavy_congestion': np.random.randint(0, 6, num_rows), # 0-5
    'signals': np.random.randint(0, 11, num_rows), # 0-10
    'emergency_density': np.random.randint(0, 4, num_rows) # 0-3
}

synthetic_df = pd.DataFrame(synthetic_data)
display(synthetic_df.head())

,incident_count,avg_speed,heavy_congestion,signals,emergency_density
0,3,32,0,6,2
1,4,59,0,6,2
2,2,54,3,8,1
3,4,11,4,10,3
4,4,50,0,10,1


## SECTION 6 — Create Target Label

We will create a binary target variable `risk` based on the specified rules:

-   `risk = 1` if `incident_count > 2` OR `heavy_congestion > 3` OR `avg_speed < 25`
-   `risk = 0` otherwise

In [6]:
# Create the 'risk' target column
synthetic_df['risk'] = 0 # Default to 0

synthetic_df.loc[
    (synthetic_df['incident_count'] > 2) |
    (synthetic_df['heavy_congestion'] > 3) |
    (synthetic_df['avg_speed'] < 25),
    'risk'
] = 1

print("Distribution of 'risk' label:")
display(synthetic_df['risk'].value_counts())
display(synthetic_df.head())

Distribution of 'risk' label:


,count
risk,
1,1876
0,624


,incident_count,avg_speed,heavy_congestion,signals,emergency_density,risk
0,3,32,0,6,2,1
1,4,59,0,6,2,1
2,2,54,3,8,1,0
3,4,11,4,10,3,1
4,4,50,0,10,1,1


## SECTION 7 — Build Dataset

We will create a Pandas DataFrame with all features and the target variable, then save it as `route_risk_dataset.csv`.

In [7]:
# The synthetic_df already contains all required columns.
dataset_path = 'route_risk_dataset.csv'
synthetic_df.to_csv(dataset_path, index=False)

print(f"Dataset saved to {dataset_path}")
display(pd.read_csv(dataset_path).head())

Dataset saved to route_risk_dataset.csv


,incident_count,avg_speed,heavy_congestion,signals,emergency_density,risk
0,3,32,0,6,2,1
1,4,59,0,6,2,1
2,2,54,3,8,1,0
3,4,11,4,10,3,1
4,4,50,0,10,1,1


## SECTION 8 — Train Machine Learning Model

We will use `RandomForestClassifier` from `scikit-learn` to train our model. The data will be split into 80% for training and 20% for testing. We'll then evaluate the model's performance.

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Define features (X) and target (y)
X = synthetic_df[['incident_count', 'avg_speed', 'heavy_congestion', 'signals', 'emergency_density']]
y = synthetic_df['risk']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {len(X_train)} samples")
print(f"Testing set size: {len(X_test)} samples")

# Initialize and train the RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
print("\n--- Model Evaluation ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Training set size: 2000 samples
Testing set size: 500 samples

--- Model Evaluation ---
Accuracy Score: 1.0000

Confusion Matrix:
[[125   0]
 [  0 375]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       125
           1       1.00      1.00      1.00       375

    accuracy                           1.00       500
   macro avg       1.00      1.00      1.00       500
weighted avg       1.00      1.00      1.00       500



## SECTION 9 — Save Model

We will save the trained `RandomForestClassifier` model using `joblib` so it can be loaded later by the FastAPI application.

In [9]:
import joblib

model_path = 'route_risk_model.pkl'
joblib.dump(model, model_path)

print(f"Trained model saved to {model_path}")

Trained model saved to route_risk_model.pkl


## SECTION 10 — Build FastAPI Application

We will create a FastAPI application within the notebook. This application will:
1. Load the trained ML model.
2. Connect to Supabase to fetch live data.
3. Derive features from the live data (or mock data if tables are empty).
4. Use the model to predict route risk.
5. Return the prediction via a `GET /route-risk` endpoint.

In [10]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib
import pandas as pd
import numpy as np
from supabase import create_client, Client
import nest_asyncio
import os
import uvicorn
import threading
import asyncio

# Apply nest_asyncio to allow asyncio to be re-entered (needed for running uvicorn in Colab)
nest_asyncio.apply()

app = FastAPI()

# Load the trained model
try:
    ml_model = joblib.load('route_risk_model.pkl')
    print("ML model loaded successfully.")
except Exception as e:
    print(f"Error loading ML model: {e}")
    ml_model = None # Handle case where model might not be loaded

# Supabase client for FastAPI (can reuse the one from above, but redefining for clarity/self-containment)
# IMPORTANT: Use your `Service Role Key` for full access.
SUPABASE_URL_API = "https://xdcsgcmgwlyndxrbkcav.supabase.co"
SUPABASE_KEY_API = userdata.get('SUPABASE_SERVICE_ROLE_KEY') or os.environ.get('SUPABASE_SERVICE_ROLE_KEY')

if not SUPABASE_KEY_API:
    print("Warning: SUPABASE_SERVICE_ROLE_KEY not found for FastAPI client.\nPlease add it.")

supab_client_api: Client = create_client(SUPABASE_URL_API, SUPABASE_KEY_API)

# Pydantic model for the response
class RouteRiskResponse(BaseModel):
    incident_count: int
    avg_speed: int
    heavy_congestion: int
    signals: int
    emergency_density: int
    route_risk: str

# Function to fetch and process live data for features
async def get_live_features():
    # --- Fetching data from Supabase --- (using async methods for FastAPI context)
    # For simplicity, if tables are empty, we will use default/mock values

    # incidents
    incidents_data = []
    try:
        res = supab_client_api.from_('incidents').select("resolved_at").execute()
        incidents_data = res.data if res.data else []
    except Exception as e:
        print(f"Error fetching incidents: {e}")
    incident_count = sum(1 for item in incidents_data if item.get('resolved_at') is None)

    # road_segments
    road_segments_data = []
    try:
        res = supab_client_api.from_('road_segments').select("current_speed, congestion").execute()
        road_segments_data = res.data if res.data else []
    except Exception as e:
        print(f"Error fetching road_segments: {e}")

    avg_speed = 0
    heavy_congestion = 0
    if road_segments_data:
        speeds = [seg['current_speed'] for seg in road_segments_data if 'current_speed' in seg]
        if speeds: avg_speed = int(np.mean(speeds))
        heavy_congestion = sum(1 for seg in road_segments_data if seg.get('congestion') == 'heavy')

    # vehicles
    vehicles_data = []
    try:
        res = supab_client_api.from_('vehicles').select("type").execute()
        vehicles_data = res.data if res.data else []
    except Exception as e:
        print(f"Error fetching vehicles: {e}")
    emergency_density = sum(1 for veh in vehicles_data if veh.get('type') in ['ambulance', 'fire', 'police'])

    # traffic_signals
    traffic_signals_data = []
    try:
        res = supab_client_api.from_('traffic_signals').select("id").execute()
        traffic_signals_data = res.data if res.data else []
    except Exception as e:
        print(f"Error fetching traffic_signals: {e}")
    signals = len(traffic_signals_data)

    # --- Mock data generation if actual data is empty for demonstration ---
    # In a real scenario, you'd ensure your Supabase tables are populated or
    # handle defaults/mocks more robustly.
    if not incidents_data and not road_segments_data and not vehicles_data and not traffic_signals_data:
        print("No live data from Supabase. Generating mock features for prediction.")
        incident_count = np.random.randint(0, 6)
        avg_speed = np.random.randint(10, 71)
        heavy_congestion = np.random.randint(0, 6)
        signals = np.random.randint(0, 11)
        emergency_density = np.random.randint(0, 4)

    return {
        "incident_count": incident_count,
        "avg_speed": avg_speed,
        "heavy_congestion": heavy_congestion,
        "signals": signals,
        "emergency_density": emergency_density,
    }

@app.get("/route-risk", response_model=RouteRiskResponse)
async def get_route_risk():
    if ml_model is None:
        raise HTTPException(status_code=500, detail="ML model not loaded.")

    # Get live features from Supabase (or mock if no data)
    features = await get_live_features()

    # Prepare features for prediction
    features_df = pd.DataFrame([features])

    # Make prediction
    prediction = ml_model.predict(features_df)[0]
    risk_label = "Risky" if prediction == 1 else "Safe"

    return RouteRiskResponse(**features, route_risk=risk_label)

print("FastAPI application defined with /route-risk endpoint.")

def run_uvicorn():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)

    loop.run_until_complete(server.serve())

print("Starting FastAPI server in a background thread...")
thread = threading.Thread(target=run_uvicorn)
thread.start()

print("FastAPI server started on http://0.0.0.0:8000. Wait for 'Uvicorn running on http://0.0.0.0:8000' message.")

ML model loaded successfully.
FastAPI application defined with /route-risk endpoint.
Starting FastAPI server in a background thread...
FastAPI server started on http://0.0.0.0:8000. Wait for 'Uvicorn running on http://0.0.0.0:8000' message.


## SECTION 11 — Run FastAPI Server

We will run the FastAPI server using `uvicorn`. We'll use `0.0.0.0:8000` so it's accessible externally (e.g., by ngrok). The server will run in the background.

In [11]:
# This cell is now empty as its content has been merged into cell 'c6cfa151'

## SECTION 12 — Make API Public Using ngrok

To make our local FastAPI server accessible over the internet, we'll use `ngrok`. You need an `NGROK_AUTH_TOKEN` from your ngrok dashboard. Store it in Colab secrets as `NGROK_AUTH_TOKEN`.

In [12]:
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError
import time

# Get ngrok auth token from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

if not NGROK_AUTH_TOKEN:
    raise ValueError("NGROK_AUTH_TOKEN not found in Colab Secrets.\nPlease add it for ngrok to function.")

try:
    # Authenticate ngrok
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("ngrok authenticated.")

    # Give a little time for uvicorn to start
    time.sleep(5)
    # Open a tunnel to the FastAPI server running on port 8000
    public_url_object = ngrok.connect(8000)
    # Extract the string URL from the NgrokTunnel object
    NGROK_PUBLIC_URL = public_url_object.public_url
    print(f"ngrok tunnel established. Public API URL: {NGROK_PUBLIC_URL}")

except PyngrokNgrokError as e:
    print(f"Error connecting to ngrok: {e}. Make sure your ngrok token is correct and the FastAPI server is running.")
    NGROK_PUBLIC_URL = None

INFO:     Started server process [5262]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


ngrok authenticated.
ngrok tunnel established. Public API URL: https://unpresidentially-nonrefracting-kyong.ngrok-free.dev


## SECTION 13 — Test the API

Finally, we will send a request to our publicly exposed FastAPI endpoint using the `ngrok` URL and display the JSON response.

In [15]:
import requests
import json

if NGROK_PUBLIC_URL:
    try:
        # Ensure NGROK_PUBLIC_URL is a string and append the endpoint path
        api_endpoint = f"{NGROK_PUBLIC_URL}/route-risk"
        print(f"Testing API endpoint: {api_endpoint}")
        response = requests.get(api_endpoint)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

        print("\n--- API Response ---")
        print(json.dumps(response.json(), indent=2))

    except requests.exceptions.ConnectionError as e:
        print(f"Connection Error: {e}. Make sure ngrok tunnel is active and FastAPI server is running.")
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
else:
    print("ngrok public URL not available. Cannot test API.")

Testing API endpoint: https://unpresidentially-nonrefracting-kyong.ngrok-free.dev/route-risk
No live data from Supabase. Generating mock features for prediction.
INFO:     35.189.171.57:0 - "GET /route-risk HTTP/1.1" 200 OK

--- API Response ---
{
  "incident_count": 4,
  "avg_speed": 23,
  "heavy_congestion": 2,
  "signals": 6,
  "emergency_density": 1,
  "route_risk": "Risky"
}
